In [38]:
# RUN THIS CELL TO INSTALL MISSING LIBRARIES
# After running, go to Kernel -> Restart to apply changes
%pip install xgboost lightgbm

Note: you may need to restart the kernel to use updated packages.


# Model Training Layer: Updated Algorithm Pipeline
This notebook implements the training pipeline using XGBoost, LightGBM, and Random Forest.

In [39]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, StandardScaler, OrdinalEncoder
from sklearn.metrics import mean_squared_error, r2_score, f1_score, accuracy_score
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

# Step 1 — Load Splits
def load_splits(data_dir='../data/'):
    train = pd.read_csv(os.path.join(data_dir, 'train.csv'))
    val = pd.read_csv(os.path.join(data_dir, 'val.csv'))
    test = pd.read_csv(os.path.join(data_dir, 'test.csv'))
    return train, val, test

train_df, val_df, test_df = load_splits()
print(f"Splits loaded. Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")

Splits loaded. Train: (922424, 23), Val: (197663, 23), Test: (197663, 23)


### Step 2 — Target Engineering (Binary Classification for A & B)

In [40]:
# Model A Target: has_wait
for df in [train_df, val_df, test_df]:
    df['has_wait'] = (df['estimated_wait_time_mins'] > 0).astype(int)

# Model B Target: high_utilization
threshold = 0.7
for df in [train_df, val_df, test_df]:
    df['high_utilization'] = (df['utilization_rate'] >= threshold).astype(int)

print("Target 'high_utilization' created (Threshold=0.7). Class Distribution (Train):")
print(train_df['high_utilization'].value_counts(normalize=True))

Target 'high_utilization' created (Threshold=0.7). Class Distribution (Train):
high_utilization
0    0.79468
1    0.20532
Name: proportion, dtype: float64


### Step 3 — Modular Evaluation Functions

In [41]:
def print_regression_metrics(model, name, X_train, y_train, X_val, y_val, X_test, y_test, y_pred_test_override=None):
    print(f"\n--- {name} Performance Metrics ---")
    summary = {}
    sets = [('Train', X_train, y_train), ('Val', X_val, y_val), ('Test', X_test, y_test)]
    
    for s_name, X, y in sets:
        preds = model.predict(X)
        if s_name == 'Test' and y_pred_test_override is not None:
            preds = y_pred_test_override
            
        rmse = np.sqrt(mean_squared_error(y, preds))
        r2 = r2_score(y, preds)
        summary[s_name] = {'RMSE': rmse, 'R2': r2}
    
    results_df = pd.DataFrame(summary).T
    print(results_df)

def print_classification_metrics(model, name, X_train, y_train, X_val, y_val, X_test, y_test):
    print(f"\n--- {name} Performance Metrics ---")
    summary = {}
    sets = [('Train', X_train, y_train), ('Val', X_val, y_val), ('Test', X_test, y_test)]
    
    for s_name, X, y in sets:
        preds = model.predict(X)
        f1 = f1_score(y, preds, average='weighted')
        acc = accuracy_score(y, preds)
        summary[s_name] = {'F1_Weighted': f1, 'Accuracy': acc}
    
    results_df = pd.DataFrame(summary).T
    print(results_df)

### ── Model A — Queue Probability (has_wait) ──────────────────

In [42]:
# Features (Refined to only 3 features)
features_a = ['ports_out_of_service', 'utilization_rate', 'traffic_congestion_index']

# Data splits
X_train_a, y_train_a = train_df[features_a], train_df['has_wait']
X_val_a,   y_val_a   = val_df[features_a],   val_df['has_wait']
X_test_a,  y_test_a  = test_df[features_a],  test_df['has_wait']

# Pipeline with XGBoost Classifier
model_a = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler()),
    ('classifier', XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, 
                                 use_label_encoder=False, eval_metric='logloss', random_state=42))
])

# Train
model_a.fit(X_train_a, y_train_a)

# Evaluate
print_classification_metrics(model_a, 'Model A', X_train_a, y_train_a, X_val_a, y_val_a, X_test_a, y_test_a)


--- Model A Performance Metrics ---
       F1_Weighted  Accuracy
Train     0.980438  0.979090
Val       0.980688  0.979369
Test      0.981121  0.979860


### ── Model B — High Utilization (high_utilization) ──────────────────

In [43]:
# Features
features_b = ['hour_of_day', 'day_of_week', 'traffic_congestion_index', 'ports_occupied', 'ports_total', 'ports_out_of_service']

# Data splits
X_train_b, y_train_b = train_df[features_b], train_df['high_utilization']
X_val_b,   y_val_b   = val_df[features_b],   val_df['high_utilization']
X_test_b,  y_test_b  = test_df[features_b],  test_df['high_utilization']

# Pipeline with LightGBM Classifier
model_b = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler()),
    ('classifier', LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, random_state=42))
])

# Train
model_b.fit(X_train_b, y_train_b)

# Evaluate
print_classification_metrics(model_b, 'Model B', X_train_b, y_train_b, X_val_b, y_val_b, X_test_b, y_test_b)


--- Model B Performance Metrics ---
       F1_Weighted  Accuracy
Train     0.943690  0.944907
Val       0.943122  0.944314
Test      0.942683  0.943899


### ── Model C — Session Duration (avg_session_duration_mins) ──────────

In [44]:
# Features (Refined to non-zero importance only)
cat_features_c = ['charger_type']
num_features_c = ['power_output_kw']
features_c = num_features_c + cat_features_c

# Data splits
X_train_c, y_train_c = train_df[features_c], train_df['avg_session_duration_mins']
X_val_c,   y_val_c   = val_df[features_c],   val_df['avg_session_duration_mins']
X_test_c,  y_test_c  = test_df[features_c],  test_df['avg_session_duration_mins']

# Pipeline with Random Forest Regressor
preprocessor_c = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', RobustScaler())]), num_features_c),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))]), cat_features_c)
])

model_c = Pipeline([
    ('preprocessor', preprocessor_c),
    ('regressor', RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42))
])

# Train
model_c.fit(X_train_c, y_train_c)

# Evaluate
print_regression_metrics(model_c, 'Model C', X_train_c, y_train_c, X_val_c, y_val_c, X_test_c, y_test_c)


--- Model C Performance Metrics ---
            RMSE        R2
Train  24.704606  0.883507
Val    24.633196  0.884012
Test   24.653568  0.883602


### ── Model D — Current Price (current_price) ──────────────────

In [45]:
# Features
cat_features_d = ['charger_type', 'pricing_type', 'network']
num_features_d = ['hour_of_day', 'is_peak_hour', 'utilization_rate', 'power_output_kw', 'ports_total']
features_d = num_features_d + cat_features_d

# Data splits
X_train_d, y_train_d = train_df[features_d], train_df['current_price']
X_val_d,   y_val_d   = val_df[features_d],   val_df['current_price']
X_test_d,  y_test_d  = test_df[features_d],  test_df['current_price']

# Pipeline with XGBoost Regressor
preprocessor_d = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_features_d),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))]), cat_features_d)
])

model_d = Pipeline([
    ('preprocessor', preprocessor_d),
    ('regressor', XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1, random_state=42))
])

# Train
model_d.fit(X_train_d, y_train_d)

# Apply deterministic rule (Free stations) for evaluation
def apply_free_rule(df, preds):
    return np.where(df['pricing_type'] == 'free', 0.0, np.clip(preds, 0, None))

preds_d_test = apply_free_rule(test_df, model_d.predict(X_test_d))

# Evaluate
print_regression_metrics(model_d, 'Model D', X_train_d, y_train_d, X_val_d, y_val_d, X_test_d, y_test_d, y_pred_test_override=preds_d_test)


--- Model D Performance Metrics ---
           RMSE        R2
Train  0.012939  0.993404
Val    0.012978  0.993323
Test   0.012829  0.993528


### Step 5 — Save All Models

In [46]:
output_path = '../models/'
os.makedirs(output_path, exist_ok=True)

joblib.dump(model_a, os.path.join(output_path, 'wait_time_model.pkl'))
joblib.dump(model_b, os.path.join(output_path, 'high_utilization_model.pkl'))
joblib.dump(model_c, os.path.join(output_path, 'duration_model.pkl'))
joblib.dump(model_d, os.path.join(output_path, 'price_model.pkl'))

print("\nSUCCESS: All models (A, B, C, D) have been trained with new algorithms, evaluated, and saved to disk.")


SUCCESS: All models (A, B, C, D) have been trained with new algorithms, evaluated, and saved to disk.
